# 04 AIFS / ECMWF Output Check

This notebook tests whether AIFS / ECMWF forecast outputs can be used as inputs for the weather forecasting and Polymarket trading project.

The aim is not to train or run an AI weather model locally. The aim is to check whether precomputed forecast outputs can be retrieved in a form that is useful for the empirical pipeline.

The target pipeline is:

`forecast run time -> forecast lead time -> 2m temperature forecast -> city-level daily maximum -> predictive distribution -> Polymarket temperature-bin probabilities`

The main variable of interest is 2m temperature, because the initial Polymarket examples are based on daily maximum temperature.

## 1. Access routes tested

This notebook checks three practical routes:

1. **Direct ECMWF Open Data retrieval using `ecmwf-opendata`**  
   This is the preferred route if AIFS or ECMWF forecast fields can be downloaded directly.

2. **AIFS Single / AIFS ENS retrieval attempts**  
   These tests check whether AI-weather forecast products can be downloaded without running the models locally.

3. **Open-Meteo ECMWF API fallback**  
   This is not the same as downloading raw AIFS fields, but it provides a convenient point-forecast API and is useful for early city-level prototyping.

In [3]:
import os
import sys
import json
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import requests
import pandas as pd

RAW_DIR = Path("../data/raw/aifs_ecmwf")
PROCESSED_DIR = Path("../data/processed/aifs_ecmwf")

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

def get_json(url, params=None, timeout=30):
    response = requests.get(url, params=params, timeout=timeout)
    print("URL:", response.url)
    print("Status code:", response.status_code)
    response.raise_for_status()
    return response.json()

def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def summarise_file(path):
    path = Path(path)
    return {
        "exists": path.exists(),
        "size_mb": round(path.stat().st_size / (1024 * 1024), 3) if path.exists() else None,
    }

## 2. Install and import ECMWF open-data client

The `ecmwf-opendata` package provides a Python interface for ECMWF Open Data. If it is not available in the current environment, it is installed below.

In [5]:
try:
    from ecmwf.opendata import Client
    print("ecmwf-opendata is already installed.")
except ImportError:
    print("Installing ecmwf-opendata...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ecmwf-opendata"])
    from ecmwf.opendata import Client
    print("ecmwf-opendata installed and imported.")

ecmwf-opendata is already installed.


## 3. Candidate locations

These are the same approximate locations used in the earlier weather-data and mini-pipeline notebooks. They are sufficient for checking whether global forecast outputs can later be mapped to city-level inputs.

In [7]:
locations = pd.DataFrame([
    {
        "city": "Hong Kong",
        "latitude": 22.3022,
        "longitude": 114.1746,
        "timezone": "Asia/Hong_Kong",
        "notes": "Approximate coordinate near Hong Kong Observatory."
    },
    {
        "city": "London",
        "latitude": 51.5053,
        "longitude": 0.0553,
        "timezone": "Europe/London",
        "notes": "Approximate coordinate for London City Airport."
    },
    {
        "city": "New York City",
        "latitude": 40.7769,
        "longitude": -73.8740,
        "timezone": "America/New_York",
        "notes": "Approximate coordinate for LaGuardia Airport."
    },
])

locations

,city,latitude,longitude,timezone,notes
0,Hong Kong,22.3022,114.1746,Asia/Hong_Kong,Approximate coordinate near Hong Kong Observat...
1,London,51.5053,0.0553,Europe/London,Approximate coordinate for London City Airport.
2,New York City,40.7769,-73.8740,America/New_York,Approximate coordinate for LaGuardia Airport.


## 4. Direct ECMWF / AIFS retrieval helper

The helper below tries several small retrieval requests and records whether each request succeeds.

The requests are deliberately small: surface-level 2m temperature, a small number of forecast steps, and one target file. This is enough to test whether the route is usable before building a full data pipeline.

In [9]:
def try_ecmwf_retrieval(
    label,
    source,
    model,
    request_kwargs,
    target_path,
):
    target_path = Path(target_path)
    
    if target_path.exists():
        target_path.unlink()
    
    print("\n" + "=" * 80)
    print("Attempt:", label)
    print("Source:", source)
    print("Model:", model)
    print("Target:", target_path)
    print("Request:", request_kwargs)
    
    try:
        client = Client(
            source=source,
            model=model,
            resol="0p25",
            infer_stream_keyword=True,
        )
        
        client.retrieve(
            target=str(target_path),
            **request_kwargs,
        )
        
        file_info = summarise_file(target_path)
        
        return {
            "label": label,
            "source": source,
            "model": model,
            "status": "success",
            "target": str(target_path),
            "file_exists": file_info["exists"],
            "file_size_mb": file_info["size_mb"],
            "error": None,
        }
        
    except Exception as e:
        file_info = summarise_file(target_path)
        
        print("Retrieval failed.")
        print(str(e)[:1200])
        
        return {
            "label": label,
            "source": source,
            "model": model,
            "status": "failed",
            "target": str(target_path),
            "file_exists": file_info["exists"],
            "file_size_mb": file_info["size_mb"],
            "error": str(e)[:1200],
        }

## 5. Try direct AIFS / ECMWF 2m-temperature downloads

This section tries a small set of possible ECMWF Open Data requests.

The results should be interpreted as an access feasibility check. A failed request does not necessarily mean the product is unavailable; it may mean that the model name, stream, type, forecast time, step, source or retention window needs adjustment.

In [11]:
retrieval_attempts = []

candidate_requests = [
    {
        "label": "AIFS Single 2m temperature, short forecast steps, ECMWF source",
        "source": "ecmwf",
        "model": "aifs-single",
        "target": RAW_DIR / "aifs_single_2t_ecmwf_sample.grib2",
        "request": {
            "time": 0,
            "step": [0, 6, 12, 24],
            "stream": "oper",
            "type": "fc",
            "levtype": "sfc",
            "param": "2t",
        },
    },
    {
        "label": "AIFS Single 2m temperature, short forecast steps, AWS source",
        "source": "aws",
        "model": "aifs-single",
        "target": RAW_DIR / "aifs_single_2t_aws_sample.grib2",
        "request": {
            "time": 0,
            "step": [0, 6, 12, 24],
            "stream": "oper",
            "type": "fc",
            "levtype": "sfc",
            "param": "2t",
        },
    },
    {
        "label": "AIFS ENS control forecast 2m temperature, ECMWF source",
        "source": "ecmwf",
        "model": "aifs-ens",
        "target": RAW_DIR / "aifs_ens_cf_2t_ecmwf_sample.grib2",
        "request": {
            "time": 0,
            "step": [24],
            "stream": "enfo",
            "type": "cf",
            "levtype": "sfc",
            "param": "2t",
        },
    },
    {
        "label": "IFS operational 2m temperature benchmark, ECMWF source",
        "source": "ecmwf",
        "model": "ifs",
        "target": RAW_DIR / "ifs_oper_2t_ecmwf_sample.grib2",
        "request": {
            "time": 0,
            "step": [0, 6, 12, 24],
            "stream": "oper",
            "type": "fc",
            "levtype": "sfc",
            "param": "2t",
        },
    },
]

for attempt in candidate_requests:
    result = try_ecmwf_retrieval(
        label=attempt["label"],
        source=attempt["source"],
        model=attempt["model"],
        request_kwargs=attempt["request"],
        target_path=attempt["target"],
    )
    retrieval_attempts.append(result)

retrieval_attempts_df = pd.DataFrame(retrieval_attempts)
retrieval_attempts_df

To ensure the stability of our systems and to preserve resources for our operational activities (network, compute, etc.), access to the open-data portal is limited to 500 simultaneous connections. This limit helps us guarantee reliable service for our operational users, especially during periods of high demand. For added reliability, the open-data is replicated across AWS, Azure, and Google Cloud. If you experience difficulties accessing the portal directly, you can also retrieve the data from these cloud platforms.



Attempt: AIFS Single 2m temperature, short forecast steps, ECMWF source
Source: ecmwf
Model: aifs-single
Target: ../data/raw/aifs_ecmwf/aifs_single_2t_ecmwf_sample.grib2
Request: {'time': 0, 'step': [0, 6, 12, 24], 'stream': 'oper', 'type': 'fc', 'levtype': 'sfc', 'param': '2t'}


<multiple>:   0%|          | 0.00/2.21M [00:00<?, ?B/s]

By downloading data from the ECMWF open data dataset, you agree to the terms: Attribution 4.0 International (CC BY 4.0). Please attribute ECMWF when downloading this data.

Attempt: AIFS Single 2m temperature, short forecast steps, AWS source
Source: aws
Model: aifs-single
Target: ../data/raw/aifs_ecmwf/aifs_single_2t_aws_sample.grib2
Request: {'time': 0, 'step': [0, 6, 12, 24], 'stream': 'oper', 'type': 'fc', 'levtype': 'sfc', 'param': '2t'}


<multiple>:   0%|          | 0.00/2.21M [00:00<?, ?B/s]


Attempt: AIFS ENS control forecast 2m temperature, ECMWF source
Source: ecmwf
Model: aifs-ens
Target: ../data/raw/aifs_ecmwf/aifs_ens_cf_2t_ecmwf_sample.grib2
Request: {'time': 0, 'step': [24], 'stream': 'enfo', 'type': 'cf', 'levtype': 'sfc', 'param': '2t'}


20260612000000-24h-enfo-cf.grib2:   0%|          | 0.00/607k [00:00<?, ?B/s]


Attempt: IFS operational 2m temperature benchmark, ECMWF source
Source: ecmwf
Model: ifs
Target: ../data/raw/aifs_ecmwf/ifs_oper_2t_ecmwf_sample.grib2
Request: {'time': 0, 'step': [0, 6, 12, 24], 'stream': 'oper', 'type': 'fc', 'levtype': 'sfc', 'param': '2t'}


<multiple>:   0%|          | 0.00/2.51M [00:00<?, ?B/s]

,label,source,model,status,target,file_exists,file_size_mb,error
0,"AIFS Single 2m temperature, short forecast ste...",ecmwf,aifs-single,success,../data/raw/aifs_ecmwf/aifs_single_2t_ecmwf_sa...,True,2.212,None
1,"AIFS Single 2m temperature, short forecast ste...",aws,aifs-single,success,../data/raw/aifs_ecmwf/aifs_single_2t_aws_samp...,True,2.212,None
2,"AIFS ENS control forecast 2m temperature, ECMW...",ecmwf,aifs-ens,success,../data/raw/aifs_ecmwf/aifs_ens_cf_2t_ecmwf_sa...,True,0.593,None
3,"IFS operational 2m temperature benchmark, ECMW...",ecmwf,ifs,success,../data/raw/aifs_ecmwf/ifs_oper_2t_ecmwf_sampl...,True,2.512,None


## 6. Optional GRIB inspection

If GRIB processing tools are available, this section tries to open any downloaded files.

If the files download successfully but cannot be opened in this environment, the access route may still be feasible; the next task would simply be to install or configure the GRIB-processing stack.

In [13]:
grib_inspection_rows = []

try:
    import xarray as xr
    
    for _, row in retrieval_attempts_df.iterrows():
        path = Path(row["target"])
        
        if not path.exists():
            grib_inspection_rows.append({
                "label": row["label"],
                "target": str(path),
                "status": "file_not_found",
                "data_variables": None,
                "coordinates": None,
                "dimensions": None,
                "error": None,
            })
            continue
        
        try:
            ds = xr.open_dataset(path, engine="cfgrib")
            grib_inspection_rows.append({
                "label": row["label"],
                "target": str(path),
                "status": "opened_with_cfgrib",
                "data_variables": list(ds.data_vars),
                "coordinates": list(ds.coords),
                "dimensions": dict(ds.dims),
                "error": None,
            })
        except Exception as e:
            grib_inspection_rows.append({
                "label": row["label"],
                "target": str(path),
                "status": "open_failed",
                "data_variables": None,
                "coordinates": None,
                "dimensions": None,
                "error": str(e)[:1200],
            })

except ImportError as e:
    grib_inspection_rows.append({
        "label": "all",
        "target": None,
        "status": "xarray_or_cfgrib_not_available",
        "data_variables": None,
        "coordinates": None,
        "dimensions": None,
        "error": str(e),
    })

grib_inspection_df = pd.DataFrame(grib_inspection_rows)
grib_inspection_df

,label,target,status,data_variables,coordinates,dimensions,error
0,"AIFS Single 2m temperature, short forecast ste...",../data/raw/aifs_ecmwf/aifs_single_2t_ecmwf_sa...,open_failed,None,None,None,unrecognized engine cfgrib must be one of: ['s...
1,"AIFS Single 2m temperature, short forecast ste...",../data/raw/aifs_ecmwf/aifs_single_2t_aws_samp...,open_failed,None,None,None,unrecognized engine cfgrib must be one of: ['s...
2,"AIFS ENS control forecast 2m temperature, ECMW...",../data/raw/aifs_ecmwf/aifs_ens_cf_2t_ecmwf_sa...,open_failed,None,None,None,unrecognized engine cfgrib must be one of: ['s...
3,"IFS operational 2m temperature benchmark, ECMW...",../data/raw/aifs_ecmwf/ifs_oper_2t_ecmwf_sampl...,open_failed,None,None,None,unrecognized engine cfgrib must be one of: ['s...


## 7. Practical point-forecast fallback: Open-Meteo ECMWF API

This section tests a practical ECMWF-style point-forecast API route.

This does not replace direct AIFS / ECMWF GRIB access, but it is useful for early city-level experiments because it returns point forecasts directly in JSON format.

In [15]:
openmeteo_ecmwf_base = "https://api.open-meteo.com/v1/ecmwf"

openmeteo_rows = []
openmeteo_status_rows = []

for _, loc in locations.iterrows():
    params = {
        "latitude": loc["latitude"],
        "longitude": loc["longitude"],
        "hourly": "temperature_2m",
        "forecast_days": 3,
        "timezone": loc["timezone"],
        "temperature_unit": "celsius",
    }
    
    try:
        data = get_json(openmeteo_ecmwf_base, params=params)
        
        raw_path = RAW_DIR / f"openmeteo_ecmwf_{loc['city'].lower().replace(' ', '_')}.json"
        save_json(data, raw_path)
        
        hourly = pd.DataFrame(data.get("hourly", {}))
        
        if len(hourly) > 0 and "temperature_2m" in hourly.columns:
            hourly["time"] = pd.to_datetime(hourly["time"])
            dailymax = hourly.groupby(hourly["time"].dt.date)["temperature_2m"].max().reset_index()
            dailymax.columns = ["date", "forecast_daily_max_temperature_2m"]
            dailymax["city"] = loc["city"]
            dailymax["latitude"] = loc["latitude"]
            dailymax["longitude"] = loc["longitude"]
            dailymax["source"] = "Open-Meteo ECMWF API"
            
            openmeteo_rows.append(dailymax)
            
            openmeteo_status_rows.append({
                "city": loc["city"],
                "status": "success",
                "n_hourly_rows": len(hourly),
                "n_daily_rows": len(dailymax),
                "raw_path": str(raw_path),
                "error": None,
            })
        else:
            openmeteo_status_rows.append({
                "city": loc["city"],
                "status": "no_hourly_temperature_returned",
                "n_hourly_rows": len(hourly),
                "n_daily_rows": 0,
                "raw_path": str(raw_path),
                "error": None,
            })
        
    except Exception as e:
        openmeteo_status_rows.append({
            "city": loc["city"],
            "status": "failed",
            "n_hourly_rows": None,
            "n_daily_rows": None,
            "raw_path": None,
            "error": str(e)[:1200],
        })

openmeteo_ecmwf_dailymax_df = (
    pd.concat(openmeteo_rows, ignore_index=True)
    if len(openmeteo_rows) > 0
    else pd.DataFrame()
)

openmeteo_status_df = pd.DataFrame(openmeteo_status_rows)

display(openmeteo_status_df)
display(openmeteo_ecmwf_dailymax_df.head(10))

URL: https://api.open-meteo.com/v1/ecmwf?latitude=22.3022&longitude=114.1746&hourly=temperature_2m&forecast_days=3&timezone=Asia%2FHong_Kong&temperature_unit=celsius
Status code: 200
URL: https://api.open-meteo.com/v1/ecmwf?latitude=51.5053&longitude=0.0553&hourly=temperature_2m&forecast_days=3&timezone=Europe%2FLondon&temperature_unit=celsius
Status code: 200
URL: https://api.open-meteo.com/v1/ecmwf?latitude=40.7769&longitude=-73.874&hourly=temperature_2m&forecast_days=3&timezone=America%2FNew_York&temperature_unit=celsius
Status code: 200


,city,status,n_hourly_rows,n_daily_rows,raw_path,error
0,Hong Kong,success,72,3,../data/raw/aifs_ecmwf/openmeteo_ecmwf_hong_ko...,None
1,London,success,72,3,../data/raw/aifs_ecmwf/openmeteo_ecmwf_london....,None
2,New York City,success,72,3,../data/raw/aifs_ecmwf/openmeteo_ecmwf_new_yor...,None


,date,forecast_daily_max_temperature_2m,city,latitude,longitude,source
0,2026-06-12,29.0,Hong Kong,22.3022,114.1746,Open-Meteo ECMWF API
1,2026-06-13,29.9,Hong Kong,22.3022,114.1746,Open-Meteo ECMWF API
2,2026-06-14,28.6,Hong Kong,22.3022,114.1746,Open-Meteo ECMWF API
3,2026-06-12,20.8,London,51.5053,0.0553,Open-Meteo ECMWF API
4,2026-06-13,20.3,London,51.5053,0.0553,Open-Meteo ECMWF API
5,2026-06-14,20.8,London,51.5053,0.0553,Open-Meteo ECMWF API
6,2026-06-12,36.4,New York City,40.7769,-73.8740,Open-Meteo ECMWF API
7,2026-06-13,32.3,New York City,40.7769,-73.8740,Open-Meteo ECMWF API
8,2026-06-14,30.6,New York City,40.7769,-73.8740,Open-Meteo ECMWF API


## 8. Retrieval summary

This table summarises the tested routes.

In [17]:
summary_rows = []

for _, row in retrieval_attempts_df.iterrows():
    summary_rows.append({
        "route": row["label"],
        "source": row["source"],
        "model": row["model"],
        "status": row["status"],
        "output_type": "GRIB forecast field",
        "file_exists": row["file_exists"],
        "file_size_mb": row["file_size_mb"],
        "main_error_or_note": row["error"],
    })

summary_rows.append({
    "route": "Open-Meteo ECMWF point forecast API",
    "source": "Open-Meteo",
    "model": "ECMWF-style point forecast API",
    "status": "success" if len(openmeteo_ecmwf_dailymax_df) > 0 else "failed",
    "output_type": "JSON point forecast",
    "file_exists": len(openmeteo_ecmwf_dailymax_df) > 0,
    "file_size_mb": None,
    "main_error_or_note": None if len(openmeteo_ecmwf_dailymax_df) > 0 else "No city-level forecast data returned.",
})

retrieval_summary_df = pd.DataFrame(summary_rows)
retrieval_summary_df

,route,source,model,status,output_type,file_exists,file_size_mb,main_error_or_note
0,"AIFS Single 2m temperature, short forecast ste...",ecmwf,aifs-single,success,GRIB forecast field,True,2.212,None
1,"AIFS Single 2m temperature, short forecast ste...",aws,aifs-single,success,GRIB forecast field,True,2.212,None
2,"AIFS ENS control forecast 2m temperature, ECMW...",ecmwf,aifs-ens,success,GRIB forecast field,True,0.593,None
3,"IFS operational 2m temperature benchmark, ECMW...",ecmwf,ifs,success,GRIB forecast field,True,2.512,None
4,Open-Meteo ECMWF point forecast API,Open-Meteo,ECMWF-style point forecast API,success,JSON point forecast,True,NaN,None


### Interpretation of retrieval results

The direct ECMWF Open Data retrieval tests were successful. AIFS Single, AIFS ENS control forecast and IFS operational 2m-temperature forecast files were downloaded as GRIB files.

The current environment could not open the GRIB files with `cfgrib`, because the GRIB-reading backend is not yet configured. This is a processing issue rather than a data-access issue. The next technical step is to install/configure the GRIB-processing stack, then extract city-level 2m-temperature values from the downloaded forecast fields.

The Open-Meteo ECMWF API also returned city-level JSON forecasts for Hong Kong, London and New York. This provides a convenient fallback route for point-forecast prototyping, but it is not equivalent to direct AIFS forecast-field retrieval.

## 9. AIFS / ECMWF to Polymarket mapping

If AIFS or ECMWF forecast fields can be retrieved, they can enter the existing Polymarket pipeline through the following structure:

1. Retrieve 2m temperature forecast fields.
2. Record forecast run time, forecast step and valid time.
3. Extract city-level values using nearest grid point or interpolation.
4. Convert the forecast time series into a local-day daily maximum temperature forecast.
5. Estimate forecast uncertainty using historical forecast errors.
6. Map the resulting predictive distribution into Polymarket temperature-bin probabilities.
7. Compare model-implied probabilities with market-implied YES prices.
8. Evaluate using Brier score, log score, calibration and later trading/backtest metrics.

This is the same pipeline developed in the earlier single-market prototype, with the forecast source replaced by AIFS / ECMWF output if the data access route is feasible.

## 10. Save outputs

The data folder is ignored by git, while the notebook records the retrieval attempts and current findings.

In [21]:
retrieval_attempts_df.to_csv(PROCESSED_DIR / "direct_aifs_ecmwf_retrieval_attempts.csv", index=False)
grib_inspection_df.to_csv(PROCESSED_DIR / "direct_aifs_ecmwf_grib_inspection.csv", index=False)
openmeteo_status_df.to_csv(PROCESSED_DIR / "openmeteo_ecmwf_api_status.csv", index=False)
retrieval_summary_df.to_csv(PROCESSED_DIR / "aifs_ecmwf_retrieval_summary.csv", index=False)

if len(openmeteo_ecmwf_dailymax_df) > 0:
    openmeteo_ecmwf_dailymax_df.to_csv(
        PROCESSED_DIR / "openmeteo_ecmwf_city_dailymax_sample.csv",
        index=False,
    )

print("Saved AIFS / ECMWF output check summaries. Raw forecast files are stored locally and ignored by git.")

Saved AIFS / ECMWF output check summaries. Raw forecast files are stored locally and ignored by git.


## Current findings

This notebook tests whether AIFS / ECMWF forecast outputs can be used as forecast inputs for the Polymarket weather-trading pipeline.

The main conclusions should be drawn from the retrieval summary table:

- A successful AIFS Single or AIFS ENS GRIB download would indicate that ECMWF Open Data is a feasible direct route for AI-weather forecast outputs.
- If the GRIB file downloads but cannot yet be opened, the remaining issue is file processing rather than data access.
- If direct AIFS retrieval fails, the recorded error messages are still useful because they identify what to ask about: model name, stream, type, open-data retention, forecast run time and product availability.
- The Open-Meteo ECMWF API is a practical fallback for city-level ECMWF-style forecasts, although it is not equivalent to direct AIFS forecast-field retrieval.
- AIFS / ECMWF retrieval was tested using current open forecast products. Historical AIFS availability for past Polymarket event dates has not yet been verified. This is an important question for the full backtest, because the dissertation will need forecasts that were available before each market’s settlement date.

Questions for discussion:

1. Which AIFS product should be prioritised: AIFS Single, AIFS ENS control forecast, or ensemble members?
2. Is historical AIFS output available for past Polymarket event dates, or mainly real-time / recent forecasts?
3. What is the easiest ECMWF route for retrieving 2m temperature by run time, valid time, lead time and location?
4. Should the empirical implementation begin with direct ECMWF GRIB fields or with an easier point-forecast API while the direct AIFS route is developed?
5. For city-level extraction, should the project use nearest grid point, interpolation, or a later station-bias-correction step?
6. If historical AIFS output is difficult to access, should the empirical backtest use accessible forecast APIs first, while using AIFS/GraphCast/FourCastNet/Pangu/Aurora as model-comparison and methodology references?